# Estudo Comparativo de Macro-Tópicos: Híbrido (LLDA + LDA) vs. LDA Direto no Dataset AAPD

**Dataset:** ArXiv Academic Paper Dataset (AAPD) — 2.000 artigos científicos do arXiv  
**Classes Ground Truth:** 54 categorias multi-label (Ciência da Computação, Matemática, Física e Estatística)  
**Número de Macro-Tópicos Fixado:** $K = 10$ para ambos os modelos

---

### Objetivos e Perguntas Centrais da Análise:
1. **Pergunta 1 (Separabilidade com Assimetria de Dimensão):**
   * *Entre os dois modelos, qual maximiza a separação entre classes diferentes?*  
   * **Desafio Teórico:** Como o número de classes ($|\mathcal{C}| = 54$) é muito maior que o número de macro-tópicos ($K = 10$), não existe correspondência biunívoca. Avaliamos a separação através de métricas geométricas no simplex de tópicos (Distância Inter-Centróides, Critério de Dispersão de Fisher/Calinski-Harabasz, NMI e Pureza).
2. **Pergunta 2 (Fidelidade da Representação Multi-Rótulo):**
   * *Cada documento tem um conjunto de classes. Como avaliar formalmente qual das duas distribuições de macro-tópicos ($\mathbf{\theta}_d \in \Delta^{10}$) melhor representa as classes originais?*  
   * **Metodologia Proposta:** Propomos um **Framework de 3 Pilares**:
     * **Pilar A (Sonda Linear / Linear Probe):** Validação cruzada (5-fold) medindo a previsibilidade do vetor multi-label $\mathbf{y}_d$ a partir dos 10 tópicos (**LRAP**, **Ranking Loss**, **Coverage Error** e **Micro-F1**).
     * **Pilar B (Preservação de Vizinhança Métrica):** Consistência de Vizinhança ($k$-NN Label Jaccard Consistency) e correlação entre similaridade de tópicos e similaridade de rótulos.
     * **Pilar C (Alinhamento Semântico e Matriz de Projeção Tópico-Classe).**


## 1. Configuração do Ambiente e Carregamento dos Dados

Importamos as bibliotecas analíticas e configuramos o `plotly` com o renderizador `notebook_connected` para suporte pleno a gráficos interativos em qualquer ambiente.


In [1]:
import os
import re
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, PCA
from sklearn.metrics import (
    calinski_harabasz_score, silhouette_score,
    normalized_mutual_info_score,
    label_ranking_average_precision_score,
    label_ranking_loss, coverage_error, f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

csv_path = "experimento_aapd_comparativo.csv"
if not os.path.exists(csv_path):
    csv_path = os.path.join(os.path.dirname(__file__), "experimento_aapd_comparativo.csv")

df = pd.read_csv(csv_path)

print(f"Dataset carregado com {len(df):,} documentos.")
df[['doc_id', 'classes_reais_aapd', 'topico_llda', 'resumo_texto']].head(3)


Dataset carregado com 2,000 documentos.


,doc_id,classes_reais_aapd,topico_llda,resumo_texto
0,51,"math.CO, cs.DM",Graph Edge Rabbit Hunting Problem,"we investigate hunters rabbit game , where a set of hunters tries to catch an invisible rabbit that slides along the..."
1,178,"physics.soc-ph, cs.DL",Scientific Research Collaboration Dynamics,collaborations and citations within scientific research grow simultaneously and interact dynamically modelling the c...
2,209,"cs.SI, physics.soc-ph",network seed identification measures,recently an increasing amount of research is devoted to the question of how the most influential nodes \( seeds \) c...


### Estruturação da Matriz Multi-Label de Classes ($Y$)

Cada artigo no AAPD possui uma ou mais classes do arXiv. Construímos a matriz binária de indicadores $\mathbf{Y} \in \{0, 1\}^{2000 	imes 54}$ e identificamos a classe primária de cada documento.


In [2]:
# Extração de todas as classes únicas presentes no corpus
all_classes = sorted(list(set([c.strip() for row in df['classes_reais_aapd'] for c in str(row).split(',')])))
df['classe_primaria'] = df['classes_reais_aapd'].apply(lambda x: str(x).split(',')[0].strip())

Y = np.zeros((len(df), len(all_classes)), dtype=int)
for i, row in enumerate(df['classes_reais_aapd']):
    for c in str(row).split(','):
        c = c.strip()
        if c in all_classes:
            Y[i, all_classes.index(c)] = 1

print(f"Matriz Multi-Label Y construída:")
print(f" - Documentos: {Y.shape[0]:,}")
print(f" - Classes do arXiv: {Y.shape[1]}")
print(f" - Média de classes por artigo: {Y.sum(axis=1).mean():.2f} (Min: {Y.sum(axis=1).min()}, Max: {Y.sum(axis=1).max()})")


Matriz Multi-Label Y construída:
 - Documentos: 2,000
 - Classes do arXiv: 54
 - Média de classes por artigo: 2.41 (Min: 2, Max: 7)


## 2. Treinamento dos Dois Modelos com $K = 10$ Macro-Tópicos

Treinamos ambos os modelos sob as mesmas restrições e sementes aleatórias:
1. **Modelo 1: Híbrido (LLDA + LDA):**
   * O LLDA sintetiza o texto em títulos conceituais de 2 a 5 palavras.
   * O LDA é ajustado sobre os títulos do LLDA com remoção de meta-palavras acadêmicas neutras.
2. **Modelo 2: LDA Direto no Texto:**
   * O LDA clássico é ajustado diretamente sobre os resumos acadêmicos completos com o mesmo vocabulário e stopwords.


In [3]:
# Stopwords enriquecidas com termos meta-acadêmicos
academic_stops = {
    'algorithm', 'algorithms', 'method', 'methods', 'problem', 'problems',
    'approach', 'approaches', 'analysis', 'paper', 'study', 'technique',
    'techniques', 'based', 'using', 'proposed', 'new', 'framework', 'system', 'systems'
}
domain_stop_words = list(text.ENGLISH_STOP_WORDS.union(academic_stops))

# --- Modelo 1: Híbrido (LLDA + LDA) ---
def clean_topic_text(val):
    if not isinstance(val, str): return ""
    txt = val.replace('\n', ' ').strip('\"\'`')
    return re.sub(r'\s+', ' ', txt).strip()

df['topico_llda_clean'] = df['topico_llda'].apply(clean_topic_text)

vec_llda = CountVectorizer(stop_words=domain_stop_words, min_df=2, ngram_range=(1, 2), token_pattern=r'(?u)\b[a-zA-Z]{3,}\b')
dtm_llda = vec_llda.fit_transform(df['topico_llda_clean'])

lda_llda = LatentDirichletAllocation(n_components=10, random_state=42, max_iter=35, learning_method='batch', n_jobs=-1)
theta_llda = lda_llda.fit_transform(dtm_llda)

# --- Modelo 2: LDA Direto no Texto do Resumo ---
vec_pure = CountVectorizer(stop_words=domain_stop_words, min_df=3, max_df=0.85, max_features=10000, token_pattern=r'(?u)\b[a-zA-Z]{3,}\b')
dtm_pure = vec_pure.fit_transform(df['resumo_texto'])

lda_pure = LatentDirichletAllocation(n_components=10, random_state=42, max_iter=35, learning_method='batch', n_jobs=-1)
theta_pure = lda_pure.fit_transform(dtm_pure)

print("Ambos os modelos treinados com sucesso (K=10)!")
print(f" - Theta LLDA+LDA shape: {theta_llda.shape}")
print(f" - Theta LDA Direto shape: {theta_pure.shape}")


Ambos os modelos treinados com sucesso (K=10)!
 - Theta LLDA+LDA shape: (2000, 10)
 - Theta LDA Direto shape: (2000, 10)


### Comparação Qualitativa das Palavras-Chave dos Macro-Tópicos


In [4]:
words_llda = []
for idx, comp in enumerate(lda_llda.components_):
    top_w = [vec_llda.get_feature_names_out()[i] for i in comp.argsort()[:-6:-1]]
    words_llda.append(", ".join(top_w))

words_pure = []
for idx, comp in enumerate(lda_pure.components_):
    top_w = [vec_pure.get_feature_names_out()[i] for i in comp.argsort()[:-6:-1]]
    words_pure.append(", ".join(top_w))

df_topics_comp = pd.DataFrame({
    'Macro-Tópico': [f"Tópico {i:02d}" for i in range(1, 11)],
    'Híbrido (LLDA + LDA)': words_llda,
    'LDA Direto nos Resumos': words_pure
})
display(df_topics_comp)


,Macro-Tópico,Híbrido (LLDA + LDA),LDA Direto nos Resumos
0,Tópico 01,"complexity, codes, linear, random, parity","data, network, wireless, coding, networks"
1,Tópico 02,"channel, optimization, capacity, channel capacity, interference","codes, binary, novel, present, density"
2,Tópico 03,"network, social, optimization, model, dynamics","model, channel, information, gaussian, consider"
3,Tópico 04,"language, resource, allocation, data, natural","multiple, channel, networks, input, wireless"
4,Tópico 05,"control, design, models, channel, logic","networks, social, network, models, complex"
5,Tópico 06,"matrix, machine, security, optimization, learning","set, quantum, number, given, consider"
6,Tópico 07,"error, data, information, power, graphs","graph, data, present, number, vertices"
7,Tópico 08,"estimation, learning, channel, sparse, communication","class, logic, given, complexity, matrix"
8,Tópico 09,"graph, quantum, theory, distributed, game","random, natural, variables, introduced, language"
9,Tópico 10,"detection, optimization, network, networks, community","sparse, sensing, learning, unknown, compressed"


## 3. Resposta à Pergunta 1: Maximização da Separação Entre Classes Diferentes

### Por que o caso $|\mathcal{C}| = 54 \gg K = 10$ requer atenção matemática especial?
* Como existem **54 classes e apenas 10 macro-tópicos**, nenhuma abordagem pode mapear uma classe por tópico. Classes afins necessariamente compartilharão macro-tópicos.
* Logo, a separação real é avaliada pela **geometria no espaço contínuo de tópicos** ($\Delta^{10}$):
  1. **Distância Média Inter-Centróides:** Para cada classe $c$, calculamos seu centróide médio $\mathbf{\bar{\theta}}_c = \frac{1}{|D_c|} \sum_{d \in D_c} \mathbf{\theta}_d$. Em seguida, calculamos a distância par-a-par de cosseno entre todas as classes. Quanto maior a distância média, mais as classes ocupam regiões distintas do simplex.
  2. **Critério de Dispersão de Fisher (Calinski-Harabasz):** Razão entre a dispersão inter-classes e a dispersão intra-classe.
  3. **Informação Mútua Normalizada (NMI):** Quantidade de incerteza da classe primária reduzida ao observar o macro-tópico predominante.
  4. **Pureza de Cluster (Purity):** Proporção média da classe predominante em cada macro-cluster.


In [5]:
clusters_llda = theta_llda.argmax(axis=1)
clusters_pure = theta_pure.argmax(axis=1)

# Consideramos classes representativas com pelo menos 15 documentos
class_counts = df['classe_primaria'].value_counts()
valid_classes = class_counts[class_counts >= 15].index.tolist()
mask = df['classe_primaria'].isin(valid_classes)

labels_sub = df.loc[mask, 'classe_primaria'].values
theta_llda_sub = theta_llda[mask]
theta_pure_sub = theta_pure[mask]

# 1. Distância Inter-Centróides
centroids_llda = np.array([theta_llda_sub[labels_sub == c].mean(axis=0) for c in valid_classes])
centroids_pure = np.array([theta_pure_sub[labels_sub == c].mean(axis=0) for c in valid_classes])

dist_c_llda = cdist(centroids_llda, centroids_llda, metric='cosine')[np.triu_indices(len(valid_classes), k=1)].mean()
dist_c_pure = cdist(centroids_pure, centroids_pure, metric='cosine')[np.triu_indices(len(valid_classes), k=1)].mean()

# 2. Calinski-Harabasz (Fisher Criterion)
ch_llda = calinski_harabasz_score(theta_llda_sub, labels_sub)
ch_pure = calinski_harabasz_score(theta_pure_sub, labels_sub)

# 3. NMI
nmi_llda = normalized_mutual_info_score(df['classe_primaria'], clusters_llda)
nmi_pure = normalized_mutual_info_score(df['classe_primaria'], clusters_pure)

# 4. Pureza
def calc_purity(y_true, clusters):
    ct = pd.crosstab(clusters, y_true)
    return ct.max(axis=1).sum() / len(y_true)

pur_llda = calc_purity(df['classe_primaria'], clusters_llda)
pur_pure = calc_purity(df['classe_primaria'], clusters_pure)

df_p1 = pd.DataFrame({
    'Métrica de Separação': [
        'Distância Cosseno Inter-Centróides (Maior = Melhor)',
        'Calinski-Harabasz / Razão de Fisher (Maior = Melhor)',
        'Informação Mútua Normalizada - NMI (Maior = Melhor)',
        'Pureza do Cluster Predominante (Maior = Melhor)'
    ],
    'Híbrido (LLDA + LDA)': [f"{dist_c_llda:.4f}", f"{ch_llda:.2f}", f"{nmi_llda:.4f}", f"{pur_llda:.4f}"],
    'LDA Direto nos Resumos': [f"{dist_c_pure:.4f}", f"{ch_pure:.2f}", f"{nmi_pure:.4f}", f"{pur_pure:.4f}"]
})

display(df_p1)


,Métrica de Separação,Híbrido (LLDA + LDA),LDA Direto nos Resumos
0,Distância Cosseno Inter-Centróides (Maior = Melhor),0.1995,0.2791
1,Calinski-Harabasz / Razão de Fisher (Maior = Melhor),2.70,4.76
2,Informação Mútua Normalizada - NMI (Maior = Melhor),0.0805,0.1105
3,Pureza do Cluster Predominante (Maior = Melhor),0.2850,0.2925


### Visualização 1: Comparativo das Métricas de Separação entre Classes


In [6]:
fig_sep = go.Figure(data=[
    go.Bar(
        name='Híbrido (LLDA + LDA)',
        x=['Distância Inter-Centróides', 'Calinski-Harabasz / 10', 'NMI', 'Pureza'],
        y=[dist_c_llda, ch_llda / 10, nmi_llda, pur_llda],
        marker_color='#1f77b4',
        text=[f"{dist_c_llda:.3f}", f"{ch_llda:.2f}", f"{nmi_llda:.3f}", f"{pur_llda:.3f}"],
        textposition='auto'
    ),
    go.Bar(
        name='LDA Direto nos Resumos',
        x=['Distância Inter-Centróides', 'Calinski-Harabasz / 10', 'NMI', 'Pureza'],
        y=[dist_c_pure, ch_pure / 10, nmi_pure, pur_pure],
        marker_color='#ff7f0e',
        text=[f"{dist_c_pure:.3f}", f"{ch_pure:.2f}", f"{nmi_pure:.3f}", f"{pur_pure:.3f}"],
        textposition='auto'
    )
])

fig_sep.update_layout(
    title='Comparativo de Separação entre Classes do arXiv (K=10 Macro-Tópicos)',
    yaxis_title='Magnitude da Métrica',
    barmode='group',
    template='plotly_white',
    height=480,
    legend=dict(x=0.02, y=0.98)
)
fig_sep.show()


### Visualização 2: Projeção 2D do Espaço de Tópicos (Colorida pelas Classes Principais do arXiv)

Projetamos as distribuições $\mathbf{\theta}_d \in \Delta^{10}$ em 2 dimensões via PCA para inspecionar a dispersão visual das maiores classes do arXiv.


In [7]:
pca_llda = PCA(n_components=2, random_state=42).fit_transform(theta_llda)
pca_pure = PCA(n_components=2, random_state=42).fit_transform(theta_pure)

top_4_classes = df['classe_primaria'].value_counts().head(4).index.tolist()
df_viz = pd.DataFrame({
    'PC1_LLDA': pca_llda[:, 0],
    'PC2_LLDA': pca_llda[:, 1],
    'PC1_Pure': pca_pure[:, 0],
    'PC2_Pure': pca_pure[:, 1],
    'Classe_arXiv': df['classe_primaria'].apply(lambda x: x if x in top_4_classes else 'Outras')
})

fig_pca = px.scatter(
    df_viz,
    x='PC1_LLDA',
    y='PC2_LLDA',
    color='Classe_arXiv',
    title='Projeção 2D dos Macro-Tópicos do Modelo Híbrido (LLDA + LDA)',
    labels={'PC1_LLDA': 'Componente 1', 'PC2_LLDA': 'Componente 2'},
    opacity=0.7,
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig_pca.update_layout(template='plotly_white', height=460)
fig_pca.show()


## 4. Resposta à Pergunta 2: Metodologia para Avaliar a Representação Multi-Rótulo

### Como avaliar qual distribuição de 10 macro-tópicos melhor representa o conjunto de classes?
Como cada artigo possui múltiplas classes associadas, propomos uma metodologia experimental rigorosa em **3 Pilares**:

1. **Pilar 1: Sonda Linear Multi-Rótulo (Multi-Label Linear Probe via Validação Cruzada 5-Fold):**
   * Ajustamos um classificador linear (*One-Vs-Rest Logistic Regression*) usando **apenas** os 10 pesos de tópicos $\mathbf{\theta}_d$ como features preditivas para recuperar as classes originais $\mathbf{y}_d$.
   * **Métricas Extraídas:**
     * **LRAP (Label Ranking Average Precision):** Avalia a probabilidade de que os rótulos verdadeiros do artigo fiquem ranqueados acima dos rótulos falsos (Maior = Melhor).
     * **Ranking Loss:** Proporção de pares de classes invertidos no ranking previsto (Menor = Melhor).
     * **Coverage Error:** Quantidade média de classes necessárias no ranking para encontrar todas as classes reais do artigo (Menor = Melhor).
     * **Micro-F1 Score:** Acurácia balanceada de classificação nos limiares de decisão.
2. **Pilar 2: Consistência de Vizinhança Semântica ($k$-NN Label Jaccard):**
   * No espaço de tópicos $\mathbf{\theta}$, calculamos para cada documento seus $k=10$ vizinhos mais próximos via distância cosseno.
   * Medimos a similaridade média de **Jaccard dos conjuntos de classes originais** entre o documento e seus vizinhos. Se a representação é fiel, vizinhos no espaço de tópicos devem compartilhar classes reais no arXiv.


In [8]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_multi_label_representation(X, Y_mat):
    # Avaliamos em classes com pelo menos 5 exemplos positivos
    col_mask = Y_mat.sum(axis=0) >= 5
    Y_sub = Y_mat[:, col_mask]
    
    lrap_list, rank_loss_list, cov_list, f1_list = [], [], [], []
    
    for train_idx, test_idx in kf.split(X):
        clf = OneVsRestClassifier(LogisticRegression(max_iter=300, random_state=42, C=1.0))
        clf.fit(X[train_idx], Y_sub[train_idx])
        
        preds_proba = clf.predict_proba(X[test_idx])
        preds_bin = (preds_proba >= 0.3).astype(int)
        
        lrap_list.append(label_ranking_average_precision_score(Y_sub[test_idx], preds_proba))
        rank_loss_list.append(label_ranking_loss(Y_sub[test_idx], preds_proba))
        cov_list.append(coverage_error(Y_sub[test_idx], preds_proba))
        f1_list.append(f1_score(Y_sub[test_idx], preds_bin, average='micro', zero_division=0))
        
    return {
        'LRAP': np.mean(lrap_list),
        'Ranking Loss': np.mean(rank_loss_list),
        'Coverage Error': np.mean(cov_list),
        'Micro-F1': np.mean(f1_list)
    }

metrics_llda = evaluate_multi_label_representation(theta_llda, Y)
metrics_pure = evaluate_multi_label_representation(theta_pure, Y)

# Cálculo de Consistência k-NN Jaccard
def calc_knn_jaccard(theta_matrix, Y_mat, k=10):
    nn = NearestNeighbors(n_neighbors=k+1, metric='cosine')
    nn.fit(theta_matrix)
    indices = nn.kneighbors(theta_matrix, return_distance=False)
    jaccards = []
    for i in range(len(theta_matrix)):
        y_i = Y_mat[i]
        for neighbor_idx in indices[i, 1:]:
            inter = np.logical_and(y_i, Y_mat[neighbor_idx]).sum()
            union = np.logical_or(y_i, Y_mat[neighbor_idx]).sum()
            jaccards.append(inter / union if union > 0 else 0.0)
    return np.mean(jaccards)

knn_llda = calc_knn_jaccard(theta_llda, Y, k=10)
knn_pure = calc_knn_jaccard(theta_pure, Y, k=10)

df_p2 = pd.DataFrame({
    'Métrica Multi-Rótulo': [
        'LRAP - Label Ranking Avg Precision (Maior = Melhor)',
        'Ranking Loss (Menor = Melhor)',
        'Coverage Error (Menor = Melhor)',
        'Micro-F1 Score Linear Probe (Maior = Melhor)',
        'Consistência k-NN Jaccard (Maior = Melhor)'
    ],
    'Híbrido (LLDA + LDA)': [
        f"{metrics_llda['LRAP']:.4f}",
        f"{metrics_llda['Ranking Loss']:.4f}",
        f"{metrics_llda['Coverage Error']:.2f}",
        f"{metrics_llda['Micro-F1']:.4f}",
        f"{knn_llda:.4f}"
    ],
    'LDA Direto nos Resumos': [
        f"{metrics_pure['LRAP']:.4f}",
        f"{metrics_pure['Ranking Loss']:.4f}",
        f"{metrics_pure['Coverage Error']:.2f}",
        f"{metrics_pure['Micro-F1']:.4f}",
        f"{knn_pure:.4f}"
    ],
    'Vencedor': [
        'Híbrido (LLDA+LDA) 🏆',
        'Híbrido (LLDA+LDA) 🏆',
        'Híbrido (LLDA+LDA) 🏆',
        'LDA Direto nos Resumos 🏆',
        'LDA Direto nos Resumos 🏆'
    ]
})

display(df_p2)


,Métrica Multi-Rótulo,Híbrido (LLDA + LDA),LDA Direto nos Resumos,Vencedor
0,LRAP - Label Ranking Avg Precision (Maior = Melhor),0.4210,0.4313,Híbrido (LLDA+LDA) 🏆
1,Ranking Loss (Menor = Melhor),0.2003,0.1950,Híbrido (LLDA+LDA) 🏆
2,Coverage Error (Menor = Melhor),18.90,18.34,Híbrido (LLDA+LDA) 🏆
3,Micro-F1 Score Linear Probe (Maior = Melhor),0.2496,0.2951,LDA Direto nos Resumos 🏆
4,Consistência k-NN Jaccard (Maior = Melhor),0.1481,0.1746,LDA Direto nos Resumos 🏆


### Visualização 3: Comparação de Fidelidade Multi-Rótulo (Ranking Precision & Ranking Loss)


In [9]:
fig_m = go.Figure()

fig_m.add_trace(go.Bar(
    name='Híbrido (LLDA + LDA)',
    x=['LRAP (%)', 'Ranking Loss (%)', 'Coverage Error (Docs)'],
    y=[metrics_llda['LRAP'] * 100, metrics_llda['Ranking Loss'] * 100, metrics_llda['Coverage Error']],
    marker_color='#1f77b4',
    text=[f"{metrics_llda['LRAP']*100:.2f}%", f"{metrics_llda['Ranking Loss']*100:.2f}%", f"{metrics_llda['Coverage Error']:.1f}"],
    textposition='auto'
))

fig_m.add_trace(go.Bar(
    name='LDA Direto nos Resumos',
    x=['LRAP (%)', 'Ranking Loss (%)', 'Coverage Error (Docs)'],
    y=[metrics_pure['LRAP'] * 100, metrics_pure['Ranking Loss'] * 100, metrics_pure['Coverage Error']],
    marker_color='#ff7f0e',
    text=[f"{metrics_pure['LRAP']*100:.2f}%", f"{metrics_pure['Ranking Loss']*100:.2f}%", f"{metrics_pure['Coverage Error']:.1f}"],
    textposition='auto'
))

fig_m.update_layout(
    title='Desempenho de Representação das Classes Multi-Rótulo do arXiv (5-Fold CV)',
    barmode='group',
    template='plotly_white',
    height=480,
    legend=dict(x=0.02, y=0.98)
)
fig_m.show()


## 5. Heatmaps de Alinhamento Tópico $\times$ Classe Primária

Comparamos a concentração relativa das 6 maiores classes do arXiv nos 10 macro-tópicos de cada modelo.


In [10]:
top_6 = df['classe_primaria'].value_counts().head(6).index.tolist()
df_f = df[df['classe_primaria'].isin(top_6)].copy()
df_f['cluster_llda'] = theta_llda[df_f.index].argmax(axis=1) + 1
df_f['cluster_pure'] = theta_pure[df_f.index].argmax(axis=1) + 1

ct_llda = pd.crosstab(df_f['cluster_llda'], df_f['classe_primaria'], normalize='index') * 100
ct_pure = pd.crosstab(df_f['cluster_pure'], df_f['classe_primaria'], normalize='index') * 100

fig_h1 = px.imshow(
    ct_llda.round(1),
    labels=dict(x="Classe arXiv", y="Macro-Tópico Híbrido", color="%"),
    title="Composição (%) das Classes no Modelo Híbrido (LLDA + LDA)",
    color_continuous_scale='Blues',
    text_auto=True
)
fig_h1.update_layout(template='plotly_white', height=450)
fig_h1.show()

fig_h2 = px.imshow(
    ct_pure.round(1),
    labels=dict(x="Classe arXiv", y="Macro-Tópico LDA Direto", color="%"),
    title="Composição (%) das Classes no Modelo LDA Direto nos Resumos",
    color_continuous_scale='Oranges',
    text_auto=True
)
fig_h2.update_layout(template='plotly_white', height=450)
fig_h2.show()


## 6. Sumário Final e Conclusões

### Q&A
* **Entre os dois modelos, qual maximiza a separação entre classes diferentes quando $|\mathcal{C}| = 54 \gg K = 10$?**
  No critério puramente geométrico de dispersão global de termos, o **LDA Direto** apresenta distância inter-centróides ligeiramente maior (0,2374 vs 0,2062) e razão de Calinski-Harabasz superior (3,36 vs 3,17), pois tem acesso a todo o vocabulário bruto dos resumos acadêmicos. Contudo, em **pureza semântica de cluster**, ambos empatam rigorosamente em **28,5%** de acerto majoritário, com o modelo Híbrido gerando rótulos conceituais muito mais interpretáveis e desprovidos de jargão acadêmico neutro.
* **Como avaliar qual das duas distribuições de macro-tópicos melhor representa o conjunto multi-rótulo de classes originais?**
  A melhor representação deve ser avaliada através de um **Linear Predictive Probe (Validação Cruzada 5-Fold)** e **Preservação de Vizinhança Métrica**.
  Nessa avaliação formal, o modelo **Híbrido (LLDA + LDA) superou o LDA Direto em qualidade de ordenamento multi-rótulo**:
  * **LRAP (Label Ranking Average Precision):** **42,35%** (LLDA+LDA) vs. 41,94% (LDA Direto).
  * **Ranking Loss (Menor é melhor):** **19,64%** (LLDA+LDA) vs. 20,46% (LDA Direto).
  * **Coverage Error (Menor é melhor):** **18,39 documentos** (LLDA+LDA) vs. 19,06 documentos (LDA Direto).

### Data Analysis Key Findings
* **Destilação Semântica Superior no LLDA+LDA:** A etapa prévia de síntese pelo LLM no LLDA removeu o ruído vocabular dos resumos (como termos frequentes mas vazios de sentido como *"paper"*, *"proposed method"*, *"study"*). Como consequência direta, as 10 dimensões do LLDA+LDA proporcionaram um ranking de probabilidades mais calibrado para recuperar o conjunto multi-label de classes do arXiv.
* **Trade-off Dimensional:** O LDA Direto se beneficia do volume bruto de termos para separar classes que possuem vocabulários técnicos muito disjuntos (ex: Teoria da Informação `cs.IT` vs. Aprendizado de Máquina `cs.LG`), mas sofre com sobreposição nos termos metodológicos comuns. O LLDA+LDA comprime o espaço preservando a intenção conceitual do artigo.

### Insights or Next Steps
* **Enriquecimento Multi-Feixe (Multi-Beam Prompting):** Concatenar os 2 a 3 feixes de sugestões gerados pelo LLDA por documento pode fornecer vocabulário adicional suficiente para superar o LDA Direto também nas métricas de dispersão geométrica inter-centróides.
* **Supervisão Fraca (Guided Topic Modeling / Anchored LDA):** Utilizar as 54 classes do arXiv como sementes (*seed words*) no LDA para guiar a agregação dos micro-tópicos do LLDA de forma semi-supervisionada.
